<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day10_CampusConnect_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install pandas


In [ ]:
from pathlib import Path
import pandas as pd
import logging
from IPython.display import display


In [ ]:
# Create folders in the temporary Colab workspace.
Path("sample_data").mkdir(exist_ok=True)
Path("output").mkdir(exist_ok=True)

rows = [
    {"Student_ID": "S101", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 9, "Classes_Held": 10},
    {"Student_ID": "S102", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 8, "Classes_Held": 10},
    {"Student_ID": "S103", "Course": "Data Analytics", "Class_Date": "2026-10-12", "Classes_Attended": 10, "Classes_Held": 10},
    {"Student_ID": "S104", "Course": "Data Analytics", "Class_Date": "2026-10-12", "Classes_Attended": 6, "Classes_Held": 10},
    {"Student_ID": "", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 7, "Classes_Held": 10},
    {"Student_ID": "S105", "Course": "Cloud Basics", "Class_Date": "2026-10-12", "Classes_Attended": 12, "Classes_Held": 10},
    {"Student_ID": "S106", "Course": "Cloud Basics", "Class_Date": "2026-10-12", "Classes_Attended": 5, "Classes_Held": 0},
    {"Student_ID": "S102", "Course": "Python", "Class_Date": "2026-10-12", "Classes_Attended": 8, "Classes_Held": 10},
    {"Student_ID": "S107", "Course": "Cloud Basics", "Class_Date": "2026-10-12", "Classes_Attended": 7, "Classes_Held": 10},
]

df_source = pd.DataFrame(rows)
input_path = Path("sample_data/attendance.csv")
df_source.to_csv(input_path, index=False)

print("Created:", input_path)
print("Number of rows:", len(df_source))
display(df_source)


Created: sample_data/attendance.csv
Number of rows: 9


,Student_ID,Course,Class_Date,Classes_Attended,Classes_Held
0,S101,Python,2026-10-12,9,10
1,S102,Python,2026-10-12,8,10
2,S103,Data Analytics,2026-10-12,10,10
3,S104,Data Analytics,2026-10-12,6,10
4,,Python,2026-10-12,7,10
5,S105,Cloud Basics,2026-10-12,12,10
6,S106,Cloud Basics,2026-10-12,5,0
7,S102,Python,2026-10-12,8,10
8,S107,Cloud Basics,2026-10-12,7,10


In [ ]:
df = pd.read_csv("sample_data/attendance.csv")

print("Rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types and missing values:")
display(df.isna().sum().to_frame("Missing values"))
display(df.head())


Rows and columns: (9, 5)

Column names:
['Student_ID', 'Course', 'Class_Date', 'Classes_Attended', 'Classes_Held']

Data types and missing values:


,Missing values
Student_ID,1
Course,0
Class_Date,0
Classes_Attended,0
Classes_Held,0


,Student_ID,Course,Class_Date,Classes_Attended,Classes_Held
0,S101,Python,2026-10-12,9,10
1,S102,Python,2026-10-12,8,10
2,S103,Data Analytics,2026-10-12,10,10
3,S104,Data Analytics,2026-10-12,6,10
4,NaN,Python,2026-10-12,7,10


In [ ]:
def run_attendance_pipeline(input_file="sample_data/attendance.csv",
                            output_dir="output"):
    input_file = Path(input_file)
    output_dir = Path(output_dir)

    if not input_file.exists():
        raise FileNotFoundError(f"Input file not found: {input_file}")

    data = pd.read_csv(input_file)

    required = [
        "Student_ID", "Course", "Class_Date",
        "Classes_Attended", "Classes_Held"
    ]
    missing_columns = set(required) - set(data.columns)
    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Convert counts to numbers; invalid text becomes NaN.
    data["Classes_Attended"] = pd.to_numeric(
        data["Classes_Attended"], errors="coerce"
    )
    data["Classes_Held"] = pd.to_numeric(
        data["Classes_Held"], errors="coerce"
    )

    # Standardize required text fields.
    for col in ["Student_ID", "Course", "Class_Date"]:
        data[col] = data[col].astype("string").str.strip()
        data[col] = data[col].replace("", pd.NA)

    # Accumulate rejection reasons instead of silently dropping records.
    reasons = pd.Series("", index=data.index, dtype="string")

    def add_reason(mask, message):
        nonlocal reasons
        reasons.loc[mask] = reasons.loc[mask].where(
            reasons.loc[mask].eq(""),
            reasons.loc[mask] + "; "
        ) + message

    missing_values = data[required].isna().any(axis=1)
    add_reason(missing_values, "Missing required value")

    invalid_held = (
        data["Classes_Held"].notna()
        & (data["Classes_Held"] <= 0)
    )
    add_reason(invalid_held, "Classes_Held must be greater than 0")

    invalid_attended = (
        data["Classes_Attended"].notna()
        & data["Classes_Held"].notna()
        & (
            (data["Classes_Attended"] < 0)
            | (data["Classes_Attended"] > data["Classes_Held"])
        )
    )
    add_reason(
        invalid_attended,
        "Classes_Attended must be between 0 and Classes_Held"
    )

    duplicate_key = data.duplicated(
        subset=["Student_ID", "Course", "Class_Date"],
        keep="first"
    )
    add_reason(
        duplicate_key,
        "Duplicate Student_ID/Course/Class_Date"
    )

    rejected_mask = reasons.ne("")
    clean = data.loc[~rejected_mask].copy()
    rejected = data.loc[rejected_mask].copy()
    rejected["Rejection_Reason"] = reasons.loc[rejected_mask]

    clean["Attendance_Percentage"] = (
        clean["Classes_Attended"] / clean["Classes_Held"] * 100
    ).round(2)

    summary = (
        clean.groupby("Course", as_index=False)
        .agg(
            Valid_Records=("Student_ID", "count"),
            Average_Attendance_Percentage=("Attendance_Percentage", "mean"),
            Total_Classes_Attended=("Classes_Attended", "sum"),
            Total_Classes_Held=("Classes_Held", "sum"),
        )
    )
    summary["Average_Attendance_Percentage"] = (
        summary["Average_Attendance_Percentage"].round(2)
    )

    output_dir.mkdir(parents=True, exist_ok=True)
    clean.to_csv(output_dir / "clean_attendance.csv", index=False)
    rejected.to_csv(output_dir / "rejected_attendance.csv", index=False)
    summary.to_csv(output_dir / "course_summary.csv", index=False)

    logging.info("Input rows: %d", len(data))
    logging.info("Clean rows: %d", len(clean))
    logging.info("Rejected rows: %d", len(rejected))
    logging.info("Output directory: %s", output_dir.resolve())
    logging.info("Pipeline completed successfully.")

    return clean, rejected, summary

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s"
)

clean, rejected, summary = run_attendance_pipeline()

print("\nCLEAN RECORDS")
display(clean)

print("\nREJECTED RECORDS")
display(rejected[["Student_ID", "Course", "Rejection_Reason"]])

print("\nCOURSE SUMMARY")
display(summary)


CLEAN RECORDS


,Student_ID,Course,Class_Date,Classes_Attended,Classes_Held,Attendance_Percentage
0,S101,Python,2026-10-12,9,10,90.0
1,S102,Python,2026-10-12,8,10,80.0
2,S103,Data Analytics,2026-10-12,10,10,100.0
3,S104,Data Analytics,2026-10-12,6,10,60.0
8,S107,Cloud Basics,2026-10-12,7,10,70.0



REJECTED RECORDS


,Student_ID,Course,Rejection_Reason
4,<NA>,Python,Missing required value
5,S105,Cloud Basics,Classes_Attended must be between 0 and Classes...
6,S106,Cloud Basics,Classes_Held must be greater than 0; Classes_A...
7,S102,Python,Duplicate Student_ID/Course/Class_Date



COURSE SUMMARY


,Course,Valid_Records,Average_Attendance_Percentage,Total_Classes_Attended,Total_Classes_Held
0,Cloud Basics,1,70.0,7,10
1,Data Analytics,2,80.0,16,20
2,Python,2,85.0,17,20


In [ ]:
assert len(df_source) == 9
assert len(clean) == 5
assert len(rejected) == 4
assert round(clean["Attendance_Percentage"].max(), 2) == 100.00

print("All basic checks passed.")
print("Input rows:", len(df_source))
print("Clean rows:", len(clean))
print("Rejected rows:", len(rejected))
print("Total rows accounted for:", len(clean) + len(rejected))


All basic checks passed.
Input rows: 9
Clean rows: 5
Rejected rows: 4
Total rows accounted for: 9


In [ ]:
clean_run_1, rejected_run_1, summary_run_1 = run_attendance_pipeline()
clean_run_2, rejected_run_2, summary_run_2 = run_attendance_pipeline()

print("Run 1 clean rows:", len(clean_run_1))
print("Run 2 clean rows:", len(clean_run_2))
print("Same clean output:", clean_run_1.equals(clean_run_2))

Run 1 clean rows: 5
Run 2 clean rows: 5
Same clean output: True


In [ ]:
try:
    run_attendance_pipeline("sample_data/does_not_exist.csv", "output/test")
except FileNotFoundError as error:
    print("Expected error caught:", error)


Expected error caught: Input file not found: sample_data/does_not_exist.csv


In [ ]:
bad_df = df_source.drop(columns=["Course"])
bad_path = Path("sample_data/attendance_missing_course.csv")
bad_df.to_csv(bad_path, index=False)

try:
    run_attendance_pipeline(str(bad_path), "output/bad_test")
except ValueError as error:
    print("Expected schema error caught:", error)


Expected schema error caught: Missing required columns: ['Course']


In [ ]:
from google.colab import files

files.download("output/clean_attendance.csv")
files.download("output/rejected_attendance.csv")
files.download("output/course_summary.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("campusconnect_outputs", "zip", "output")
files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import shutil

drive_output = Path("/content/drive/MyDrive/CampusConnect_Day10")
drive_output.mkdir(parents=True, exist_ok=True)

for file_path in Path("output").glob("*.csv"):
    shutil.copy2(file_path, drive_output / file_path.name)

print("Copied outputs to:", drive_output)


Copied outputs to: /content/drive/MyDrive/CampusConnect_Day10


In [ ]:
%pip install -q azure-ai-ml azure-identity


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.6/93.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.7/455.7 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.7/297.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import azure.ai.ml, azure.identity
print("azure-ai-ml:", azure.ai.ml.__version__)

azure-ai-ml: 1.35.2


In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DeviceCodeCredential

credential = DeviceCodeCredential()   # prints a code and a link such as microsoft.com/devicelogin

ml_client = MLClient(
    credential=credential,
    subscription_id="<your-subscription-id>",
    resource_group_name="<your-resource-group>",
    workspace_name="<your-workspace-name>",
)
print("Connected to workspace:", ml_client.workspace_name)

Connected to workspace: <your-workspace-name>


In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DeviceCodeCredential

credential = DeviceCodeCredential()   # shows a code and the link microsoft.com/devicelogin

ml_client = MLClient(
    credential=credential,
    subscription_id="<Azure subscription 1>",
    resource_group_name="sb-mazeazusrqn9-wyr6i",
    workspace_name="azureml",
)

print("Connected to workspace:", ml_client.workspace_name)

Connected to workspace: azureml


In [20]:
import logging
logging.getLogger("opentelemetry").setLevel(logging.ERROR)   # hides harmless warnings

from azure.ai.ml import MLClient
from azure.identity import DeviceCodeCredential

credential = DeviceCodeCredential()
ml_client = MLClient.from_config(credential=credential, path="config.json")

# This call really contacts Azure, so it triggers the sign-in:
ws = ml_client.workspaces.get(ml_client.workspace_name)
print("Connected to workspace:", ws.name, "| location:", ws.location)



Found the config file in: config.json


To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code EDP4GQPAZ to authenticate.
Connected to workspace: azureml | location: southindia


In [22]:
from datetime import datetime
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

MY_ID = "hs1"      # your own ID (lowercase letters/digits)
assert MY_ID.isalnum() and MY_ID.islower(), "Use lowercase letters/digits only"

data_asset = Data(
    name=f"{MY_ID}-campusconnect-attendance",
    version=datetime.now().strftime("%Y%m%d%H%M%S"),   # new version on every run
    type=AssetTypes.URI_FILE,
    path="sample_data/attendance.csv",
    description="Fictional CampusConnect attendance file",
)
registered = ml_client.data.create_or_update(data_asset)
print("Registered data asset:", registered.name, "| version:", registered.version)

Uploading attendance.csv (< 1 MB): 100%|██████████| 342/342 [00:00<00:00, 1.48kB/s]




Registered data asset: hs1-campusconnect-attendance | version: 20261015094153


In [29]:
from pathlib import Path

print("ml_client exists :", "ml_client" in globals())
print("registered exists:", "registered" in globals())
print("submitted exists :", "submitted" in globals())
print("script exists    :", Path("src/attendance_pipeline.py").exists())

ml_client exists : True
registered exists: True
submitted exists : False
script exists    : True


In [30]:
from pathlib import Path
from azure.ai.ml import command, Input, Output
from azure.ai.ml.constants import AssetTypes

assert Path("src/attendance_pipeline.py").exists(), "Run Cell 2 first (src folder is missing)"

# None = serverless compute. If your trainer gave a cluster name,
# put it here instead, e.g. "cpu-cluster".
COMPUTE_NAME = None

job = command(
    code="./src",
    command=(
        "python attendance_pipeline.py "
        "--input ${{inputs.attendance_csv}} "
        "--output ${{outputs.results}}"
    ),
    inputs={
        "attendance_csv": Input(
            type=AssetTypes.URI_FILE,
            path=f"azureml:{registered.name}:{registered.version}",
        )
    },
    outputs={"results": Output(type=AssetTypes.URI_FOLDER)},
    environment="azureml://registries/azureml/environments/sklearn-1.5/labels/latest",
    compute=COMPUTE_NAME,
    display_name=f"{MY_ID}-campusconnect-attendance",
    experiment_name="campusconnect-day10",
)

submitted = ml_client.jobs.create_or_update(job)
print("Job name  :", submitted.name)
print("Status    :", submitted.status)
print("Studio URL:", submitted.studio_url)

Uploading src (0.0 MBs): 100%|██████████| 3488/3488 [00:00<00:00, 15396.56it/s]




Job name  : busy_monkey_rsnszn455k
Status    : JobStatus.STARTING
Studio URL: https://ml.azure.com/runs/busy_monkey_rsnszn455k?wsid=/subscriptions/337f2b3a-68b6-4a2e-befd-01a13f20c1d0/resourcegroups/sb-mazeazusrqn9-wyr6i/workspaces/azureml&tid=0e22b77c-7ae6-4b55-bf27-94f160f78abb


In [32]:
for u in ml_client.compute.list_usage():
    name = getattr(u.name, "value", u.name)
    limit = getattr(u, "limit", 0) or 0
    used = getattr(u, "current_value", 0) or 0
    if "Famil" in str(name) and limit > 0:
        print(f"{name}: used {used} of {limit} cores")

In [34]:
from datetime import datetime
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

results_asset = Data(
    name=f"{MY_ID}-campusconnect-results",
    version=datetime.now().strftime("%Y%m%d%H%M%S"),
    type=AssetTypes.URI_FOLDER,
    path="output",
    description="Clean, rejected and summary files from the Colab pipeline",
)
saved = ml_client.data.create_or_update(results_asset)
print("Saved to Azure ML:", saved.name, "| version:", saved.version)

Uploading output (0.0 MBs): 100%|██████████| 835/835 [00:00<00:00, 935.56it/s]




Saved to Azure ML: hs1-campusconnect-results | version: 20261015095616


In [25]:
!pwd
!ls -l src

/content
ls: cannot access 'src': No such file or directory
